# Stackelberg : la performativité sans mystère

Une action **mauvaise, voire dominée** dans le jeu simultané peut devenir la meilleure si l'agent peut **s'y engager** avant que l'autre ne choisisse — parce que l'engagement **transforme la meilleure réponse de l'autre**. C'est le cœur de la solution de Stackelberg [von Stackelberg, 1934], et c'est aussi, déguisé en théorie des jeux, un noyau mathématique propre pour ce qu'Austin [1962] appelle le **performatif** : un énoncé (ou un acte) qui ne décrit pas le monde mais le *fait*.

```text
sigma_L  ->  BR_F(sigma_L)  ->  u_L(sigma_L, BR_F(sigma_L))
```

> **L'acte n'a plus seulement une valeur par son RÉSULTAT DIRECT. Il a une valeur parce qu'il MODIFIE L'ESPACE DE RÉPONSE D'AUTRUI.**

La condition de félicité d'Austin — ce qui distingue une annonce qui *fait* d'une annonce qui ne *dit* rien — se traduit ici en termes brutaux : **l'engagement doit être crédible**. S'il ne l'est pas, rien ne change. S'il l'est, l'annonce devient **causalement constitutive** du jeu qui suit :

```text
representation publique  ->  anticipation collective  ->  changement des actions  ->  nouvelle realite strategique
```

Ce notebook démonte la mécanique sur un jeu d'entrée sur marché, en quatre régimes calculés exactement : le jeu simultané, la mise séquentielle sans engagement, l'engagement contraignant, et l'engagement révocable — puis il **mesure** ce que coûte la crédibilité, et montre que le seuil à payer est exactement l'écart de tentation du déviateur. Une performativité sans mystère : chaque affirmation de ce notebook est un nombre calculé dans une cellule.

Le fil suivra la série [GameTheory](GameTheory-02-NormalForm.ipynb) (jeux sous forme normale, [équilibres de Nash](GameTheory-04-NashEquilibrium.ipynb), [induction à rebours](GameTheory-09-BackwardInduction.ipynb)) et rejoint le chantier « strate 7 » (vocabulaires et extensions) : l'engagement est l'archétype de l'acte qui change l'espace des actions disponibles plutôt que d'en choisir une.

In [1]:
# === Le jeu d'entree sur marche : l'entrant (E) face a l'incumbent (I) ===
# Convention : chaque case = (u_entrant, u_incumbent).
#   Entrant : 'In' (entrer) / 'Out' (rester dehors).
#   Incumbent : 'Acc' (accommoder, se partager le marche) / 'Fight' (guerre de prix).
ACTIONS_E = ['In', 'Out']
ACTIONS_I = ['Acc', 'Fight']
U = {('In',  'Acc'):   (2,  1),   # duopole accommodant
     ('In',  'Fight'): (-1, -2),  # guerre de prix : les deux perdent
     ('Out', 'Acc'):   (0,  0),   # marche vide, rien a accommoder
     ('Out', 'Fight'): (0,  3)}   # dissuasion reussie : monopole intact

def u_E(ae, ai): return U[(ae, ai)][0]
def u_I(ae, ai): return U[(ae, ai)][1]

def br_E(ai):  # meilleure reponse de l'entrant a l'action de l'incumbent
    return max(ACTIONS_E, key=lambda ae: u_E(ae, ai))

def br_I(ae):  # meilleure reponse de l'incumbent a l'action de l'entrant
    return max(ACTIONS_I, key=lambda ai: u_I(ae, ai))

print('Matrice de gains (u_entrant, u_incumbent) :')
print(f'{"":10} {"Acc":>12} {"Fight":>12}')
for ae in ACTIONS_E:
    row = ' '.join(f'{str(U[(ae, ai)]):>12}' for ai in ACTIONS_I)
    print(f'{ae:10} {row}')
print()
print("Meilleures reponses :")
for ai in ACTIONS_I:
    print(f'  BR_entrant({ai:5}) = {br_E(ai):4}   (gains E : In={u_E("In", ai):>2}, Out={u_E("Out", ai):>2})')
for ae in ACTIONS_E:
    print(f'  BR_incumbent({ae:4}) = {br_I(ae):5}   (gains I : Acc={u_I(ae, "Acc"):>2}, Fight={u_I(ae, "Fight"):>2})')

Matrice de gains (u_entrant, u_incumbent) :
                    Acc        Fight
In               (2, 1)     (-1, -2)
Out              (0, 0)       (0, 3)

Meilleures reponses :
  BR_entrant(Acc  ) = In     (gains E : In= 2, Out= 0)
  BR_entrant(Fight) = Out    (gains E : In=-1, Out= 0)
  BR_incumbent(In  ) = Acc     (gains I : Acc= 1, Fight=-2)
  BR_incumbent(Out ) = Fight   (gains I : Acc= 0, Fight= 3)


### Lecture : la valeur de Fight vit dans la colonne Out

Regardez les deux meilleures réponses de l'incumbent : contre une **entrée**, il accommodne (1 > −2) ; contre l'**abstention**, il « combat » (3 > 0) — combattre un marché vide ne coûte rien et laisse le monopole intact. L'action `Fight` n'est donc pas dominée dans l'absolu, mais elle est **strictement dominée conditionnellement à l'entrée** : sur la colonne `In`, elle vaut −2 contre 1 pour `Acc`. Toute sa valeur (3) vit dans la colonne `Out` — une colonie de gains qui n'existe que si l'entrant reste dehors.

C'est la tension de tout le notebook : `Fight` est l'action qu'il *faudrait* pouvoir choisir, précisément dans le monde où elle ne coûte rien — et l'entrant le sait. La question de Stackelberg est de savoir si l'incumbent peut **faire exister** la colonne `Out`. La question d'Austin est de savoir à quel prix l'annonce « je combattrai » devient un fait plutôt qu'un souffle.

In [2]:
# === Regime 1 : le jeu simultane -- equilibres de Nash en actions pures ===
nes = [(ae, ai) for ae in ACTIONS_E for ai in ACTIONS_I
       if ae == br_E(ai) and ai == br_I(ae)]
print('Equilibres de Nash (actions pures) du jeu simultane :')
for ae, ai in nes:
    print(f'  ({ae}, {ai})  ->  u_E = {u_E(ae, ai):>2}, u_I = {u_I(ae, ai):>2}')

# === Regime 2 : mise sequentielle SANS engagement (l'entrant joue le premier) ===
# Induction a rebours : l'incumbent repond, l'entrant anticipe la reponse.
print()
print('Mise sequentielle (entrant puis incumbent), induction a rebours :')
for ae in ACTIONS_E:
    print(f'  si E joue {ae:4} : I repond {br_I(ae):5} -> u_E = {u_E(ae, br_I(ae)):>2}')
seq_ae = max(ACTIONS_E, key=lambda ae: u_E(ae, br_I(ae)))
print(f'  -> issue sequentielle : ({seq_ae}, {br_I(seq_ae)}), u_E = {u_E(seq_ae, br_I(seq_ae))}, u_I = {u_I(seq_ae, br_I(seq_ae))}')

Equilibres de Nash (actions pures) du jeu simultane :
  (In, Acc)  ->  u_E =  2, u_I =  1
  (Out, Fight)  ->  u_E =  0, u_I =  3

Mise sequentielle (entrant puis incumbent), induction a rebours :
  si E joue In   : I repond Acc   -> u_E =  2
  si E joue Out  : I repond Fight -> u_E =  0
  -> issue sequentielle : (In, Acc), u_E = 2, u_I = 1


### Lecture : deux équilibres simultanés, un seul survit à la séquentialisation

Le jeu simultané porte **deux** équilibres de Nash : `(In, Acc)` où l'entrant vient et se fait accueillir (l'incumbent touche 1), et `(Out, Fight)` où l'entrant reste dehors face à la menace (l'incumbent touche 3). Le second vaut trois fois le premier pour l'incumbent — mais regardez ce que la mise séquentielle en fait : dès que l'entrant peut *tester* la menace en jouant le premier, l'induction à rebours la dissout. L'incumbent qui fait face à une entrée réelle accommode (1 > −2), l'entrant le sait, entre, et l'issue retombe sur `(In, Acc)` : **u_I = 1**.

L'équilibre `(Out, Fight)` n'était pas faux — c'est un équilibre de Nash parfaitement valide de la forme normale — il était **incroyable** : sa valeur repose sur une action (`Fight` face à `In`) que l'incumbent ne choisirait jamais au moment de la choisir. La différence 3 − 1 = **2** n'est pas perdue : elle est *en attente*. C'est exactement le montant qu'un engagement contraignant peut débloquer — et que rien d'autre ne peut.

In [3]:
# === Regime 3 : l'engagement CONTRAIGNANT (l'incumbent joue le premier, de facon liante) ===
# Pour chaque action a laquelle l'incumbent peut s'engager, l'entrant best-repond, l'incumbent encaisse.
print('Engagement contraignant : tableau (engagement -> reponse de E -> gain de I)')
commit = {}
for ai in ACTIONS_I:
    r = br_E(ai)
    commit[ai] = u_I(r, ai)
    print(f'  s engager a {ai:5} -> E repond {r:4} -> u_I = {u_I(r, ai):>2}')
best_ai = max(ACTIONS_I, key=lambda ai: commit[ai])
gain_seq = u_I(seq_ae, br_I(seq_ae))
print(f'  -> engagement optimal : {best_ai} (u_I = {commit[best_ai]}), contre {gain_seq} sans engagement : delta = {commit[best_ai] - gain_seq:+d}')

# === Engagement MIXTE : l'incumbent s'engage a une probabilite p de Fight ===
print()
print('Engagement mixte (p = P(Fight)) : meilleure reponse de E et gain de I')
print(f'{"p":>5} {"BR_E":>6} {"u_I":>8}')
for k in range(21):
    p = k / 20
    in_pay = 2 - 3 * p        # u_E(In) si I mixe (p Fight, 1-p Acc)
    r = 'In' if in_pay > 0 else ('Out' if in_pay < 0 else 'indifferent')
    if r == 'In':
        val = 1 - 3 * p       # u_I si E entre
    else:
        val = 3 * p           # u_I si E reste dehors (pire cas a l indifferenciation)
    print(f'{p:>5.2f} {r:>10} {val:>8.2f}')

Engagement contraignant : tableau (engagement -> reponse de E -> gain de I)
  s engager a Acc   -> E repond In   -> u_I =  1
  s engager a Fight -> E repond Out  -> u_I =  3
  -> engagement optimal : Fight (u_I = 3), contre 1 sans engagement : delta = +2

Engagement mixte (p = P(Fight)) : meilleure reponse de E et gain de I
    p   BR_E      u_I
 0.00         In     1.00
 0.05         In     0.85
 0.10         In     0.70
 0.15         In     0.55
 0.20         In     0.40
 0.25         In     0.25
 0.30         In     0.10
 0.35         In    -0.05
 0.40         In    -0.20
 0.45         In    -0.35
 0.50         In    -0.50
 0.55         In    -0.65
 0.60         In    -0.80
 0.65         In    -0.95
 0.70        Out     2.10
 0.75        Out     2.25
 0.80        Out     2.40
 0.85        Out     2.55
 0.90        Out     2.70
 0.95        Out     2.85
 1.00        Out     3.00


### Lecture : l'action sous-optimale devient l'engagement unique optimal

Le tableau de l'engagement est la cellule centrale du notebook :

| engagement de I | réponse de E | gain de I |
|---|---|---|
| `Acc` | `In` | 1 |
| **`Fight`** | **`Out`** | **3** |

S'engager à `Acc` ne change rien (l'entrant vient, gain 1). S'engager à `Fight` **transforme la meilleure réponse de l'entrant** — face à un adversaire lié à la guerre de prix, `Out` (0) domine `In` (−1) — et rapporte 3. Le delta est **+2**, exactement l'écart entre les deux équilibres du jeu simultané : l'engagement n'invente pas de valeur, il **sélectionne** l'équilibre que la menace incroyable ne pouvait pas tenir.

Et le point décisif, celui qui donne son titre à l'issue : `Fight` était **strictement dominée conditionnellement à l'entrée** (−2 < 1 sur la colonne `In`). Elle devient l'engagement *unique* optimal — non parce que sa valeur directe aurait changé (combattre une entrée vaut toujours −2), mais parce que **commettre la transforme en device de sélection** : l'acte ne vaut pas par son résultat direct, il vaut par la modification de l'espace de réponse d'autrui. C'est le sens exact de la chaîne `sigma_L -> BR_F(sigma_L) -> u_L` ouverte en tête de notebook.

L'engagement **mixte** referme la porte d'une évasion : aucun mixage ne bat le sommet pur. Tant que p < 2/3, l'entrant entre et le gain de l'incumbent décroît (1 − 3p) ; dès que p ≥ 2/3, l'entrant sort et le gain plafonne à 3p, maximal à p = 1. La valeur d'engagement de ce jeu est un **sommet du simplexe**, pas un intérieur — la nuance honnête étant qu'il existe d'autres jeux (les jeux d'inspection, où le contrôleur doit rester imprévisible) où c'est l'intérieur qui gagne : le phénomène « mixer bat commettre » existe, ce n'est simplement pas celui d'ici.

## Exercice 1 : le duel publicitaire — équilibres et action candidate

Le jeu de ce notebook n'est pas le seul où vit un engagement. Soit le duel publicitaire suivant : deux firmes A (leader potentiel) et B ; A choisit `PasCampagne` / `Campagne`, B choisit `Reste` / `Attaque`. Gains `(u_A, u_B)` : `('PasCampagne','Reste'): (3, 3)`, `('PasCampagne','Attaque'): (1, 4)`, `('Campagne','Reste'): (2, 0)`, `('Campagne','Attaque'): (0, -1)`.

Calculez les meilleures réponses, listez **tous** les équilibres de Nash en actions pures du jeu simultané, puis l'issue de la mise séquentielle où B joue le premier. Identifiez enfin l'action de A qui est sous-optimale conditionnellement à `Attaque` — la candidate au rôle que `Fight` joue ci-dessus.

```python
# TODO etudiant
# Etape 1 : coder la matrice U_EXO1 et les fonctions br_A / br_B (modeles de la cellule 2).
# Etape 2 : enumerer les equilibres de Nash (modele de la cellule 3).
# Etape 3 : induction a rebours (B premier) et identification de l'action candidate.
result_exo1 = None
print('Exercice a completer : equilibres du duel publicitaire et action candidate.')
```

In [4]:
# EXERCICE 1 : equilibres du duel publicitaire, mise sequentielle B-premier,
# action sous-optimale conditionnellement (la candidate a l'engagement).
# Indice : repris cellule 2 pour la matrice, cellule 3 pour les equilibres.
# Etape 1 : U_EXO1 = {('PasCampagne','Reste'): (3,3), ...} + br_A / br_B.
# Etape 2 : enumeration des NE par double best-response.
# Etape 3 : induction a rebours (B joue, A repond) -> issue, puis comparer a chaque NE.
result_exo1 = None  # TODO etudiant
print('Exercice a completer : equilibres du duel publicitaire et action candidate.')

Exercice a completer : equilibres du duel publicitaire et action candidate.


In [5]:
# === Regime 4 : l'annonce REVOCABLE -- l'engagement sans le lien ===
# Arbre a trois etages : I annonce (sans effet lie), puis E choisit, puis I choisit LIBREMENT.
# Toutes les annonces menent au meme sous-arbre : seule compte la decision finale de I.
def sous_arbre_final(ae):
    # Au dernier etage, I choisit librement entre Acc et Fight.
    return {ai: U[(ae, ai)] for ai in ACTIONS_I}

def induction_rebours(arbre_bas):
    # Etage du bas : I (indice 1) choisit dans {Acc: ..., Fight: ...}
    bas = {ae: max(sous.items(), key=lambda kv: kv[1][1]) for ae, sous in arbre_bas.items()}
    # Etage du haut : E (indice 0) choisit In / Out en anticipant bas
    issue = max(bas.items(), key=lambda kv: kv[1][1][0])
    return issue  # (action_E, (action_I, (u_E, u_I)))

arbre = {ae: sous_arbre_final(ae) for ae in ACTIONS_E}
action_E, (action_I, payoffs) = induction_rebours(arbre)
print('Annonce revocable (aucun lien), induction a rebours sur tout l arbre :')
for ae in ACTIONS_E:
    rep_I = max(ACTIONS_I, key=lambda ai: u_I(ae, ai))
    print(f'  si E joue {ae:4} : I choisit en dernier {rep_I:5} -> {U[(ae, rep_I)]}')
print(f'  -> issue : ({action_E}, {action_I})  ->  u_E = {payoffs[0]}, u_I = {payoffs[1]}')
print(f'  -> le gain d engagement (3) retombe a {payoffs[1]} : dissous, quelle que soit l annonce.')

Annonce revocable (aucun lien), induction a rebours sur tout l arbre :
  si E joue In   : I choisit en dernier Acc   -> (2, 1)
  si E joue Out  : I choisit en dernier Fight -> (0, 3)
  -> issue : (In, Acc)  ->  u_E = 2, u_I = 1
  -> le gain d engagement (3) retombe a 1 : dissous, quelle que soit l annonce.


### Lecture : la condition de félicité — l'annonce n'est pas l'engagement

Retirez le lien (l'irrévocabilité) et tout s'effondre dans l'ordre exact prédit par la théorie : au dernier étage, l'incumbent face à une entrée **réelle** accommodne (1 > −2) — au moment de choisir, la menace n'a plus de valeur ; l'entrant, qui fait cette induction à rebours autant que nous, entre (2 > 0) ; et le gain de l'engagement **3 → 1**. Aucune annonce, si forte soit-elle, n'y change rien : l'issue est **indépendante de l'annonce**. C'est le théorème du *cheap talk* dans sa version la plus plate — quand parler ne coûte rien et n'engage à rien, parler ne fait rien.

C'est ici que la condition de félicité d'Austin cesse d'être une métaphore littéraire. « Je combattrai » n'est pas un énoncé performatif en soi ; il ne le devient que si l'énonciation **modifie les gains ou les contraintes du locuteur** — si se dédire coûte. L'annonce révocable est un énoncé *constatif* déguisé : elle décrit une intention que tout le monde sait révisable, et l'anticipation collective la traite comme telle. La performativité n'est pas dans les mots : elle est dans **l'architecture du jeu que les mots modifient**.

In [6]:
# === La credibilite au prix exact : le seuil de la caution (bond) ===
# L'incumbent signe un contrat : accommoder APRES avoir promis de combattre coute s (penalite).
# Payoff modifie : u_I(In, Acc) = 1 - s. Le reste est inchange.
def equilibre_avec_bond(s):
    # Dernier etage : I face a In compare Acc (1-s) a Fight (-2) ; face a Out : Fight (3 > 0).
    rep_in = 'Acc' if 1 - s > -2 else 'Fight'
    rep_out = 'Fight'
    # L'entrant anticipe :
    if rep_in == 'Fight':
        ae = 'Out'  # entrer ne rapporte que -1
        return (ae, 'Fight'), U[('Out', 'Fight')]
    ae = 'In'
    return (ae, rep_in), (2, 1 - s)

print(f'{"s (caution)":>12} {"issue":>16} {"u_I":>8}  regime')
for k in range(9):
    s = k * 0.5
    issue, pay = equilibre_avec_bond(s)
    regime = 'annonce dissoute (non credible)' if issue[1] == 'Acc' else 'dissuasion tenue (credible)'
    print(f'{s:>12.1f} {str(issue):>16} {pay[1]:>8.2f}  {regime}')
print()
_, pay_3 = equilibre_avec_bond(3.0)
ecart_tentation = u_I('In', 'Acc') - u_I('In', 'Fight')
print(f'Seuil de credibilite : s* = {ecart_tentation} (ecart de tentation 1 - (-2)).')
print(f'Au seuil s = 3 : u_I = {pay_3[1]} -- la pleine valeur d engagement est restauree.')

 s (caution)            issue      u_I  regime
         0.0    ('In', 'Acc')     1.00  annonce dissoute (non credible)
         0.5    ('In', 'Acc')     0.50  annonce dissoute (non credible)
         1.0    ('In', 'Acc')     0.00  annonce dissoute (non credible)
         1.5    ('In', 'Acc')    -0.50  annonce dissoute (non credible)
         2.0    ('In', 'Acc')    -1.00  annonce dissoute (non credible)
         2.5    ('In', 'Acc')    -1.50  annonce dissoute (non credible)
         3.0 ('Out', 'Fight')     3.00  dissuasion tenue (credible)
         3.5 ('Out', 'Fight')     3.00  dissuasion tenue (credible)
         4.0 ('Out', 'Fight')     3.00  dissuasion tenue (credible)

Seuil de credibilite : s* = 3 (ecart de tentation 1 - (-2)).
Au seuil s = 3 : u_I = 3 -- la pleine valeur d engagement est restauree.


### Lecture : le prix exact de la félicité — la caution doit couvrir la tentation

La caution transforme l'architecture : accommoder après avoir promis coûte désormais `1 − s`. Le balayage le montre sans zone grise : en dessous de `s* = 3`, l'entrant entre, l'incumbent accommodne en payant la peine, et le gain reste celui du jeu de base ; à partir de `s* = 3` (égalité exacte au seuil, stricte au-dessus), combattre devient la meilleure réponse **même face à une entrée réelle**, l'entrant le sait, reste dehors, et la valeur d'engagement **3** est intégralement restaurée. La condition de félicité d'Austin est devenue un nombre.

Et ce nombre a une forme remarquable : `s*` est exactement **l'écart de tentation** — la différence entre céder (1) et tenir (−2) au moment de la tester. La caution n'a pas besoin d'être aussi grande que le gain de l'engagement ni que le dommage de la guerre ; elle doit couvrir précisément ce que coûte le fait de dévier *au moment où dévier tenterait*. C'est la définition opérationnelle de la crédibilité : **crédible = dévier ne rapporte plus**, et le prix à payer pour y arriver se calcule sur la matrice. La « réputation », la « détermination », les métaphores volontaristes de la dissuasion se réduisent ici à cette inégalité.

## Exercice 2 : le couple (engagement, gain) sur VOTRE matrice

Reprenez le duel publicitaire de l'Exercice 1. Construisez le tableau de l'engagement contraignant : pour **chacune** des deux actions pures de A, calculez la meilleure réponse de B et le gain de A qui en résulte. Produisez le **couple** `(stratégie d'engagement, gain)` optimal, et comparez-le au gain de A à l'issue séquentielle sans engagement — le delta d'engagement en un chiffre. Vérifiez enfin le mixte sur une grille `p ∈ [0, 1]` (pas de 0,05) : le sommet est-il pur, comme dans le jeu d'entrée, ou intérieur ?

```python
# TODO etudiant
# Etape 1 : pour chaque action de A, BR de B puis gain de A (modele : cellule 5).
# Etape 2 : argmax -> le couple (engagement, gain).
# Etape 3 : balayage mixte p par pas de 0.05 -> sommet pur ou interieur ?
result_exo2 = None
print('Exercice a completer : couple (engagement, gain) et delta sur le duel publicitaire.')
```

In [7]:
# EXERCICE 2 : couple (engagement, gain) + delta + test du mixte sur U_EXO1.
# Indice : deux lignes de table {action: gain}, un argmax, une boucle 21 points.
# Etape 1 : tableau engagement (BR de B -> gain de A).
# Etape 2 : argmax = le couple a produire ; delta contre l issue sequentielle B-premier.
# Etape 3 : balayage p = P(seconde action de A) -> valeur d engagement en fonction de p.
result_exo2 = None  # TODO etudiant
print('Exercice a completer : couple (engagement, gain) et delta sur le duel publicitaire.')

Exercice a completer : couple (engagement, gain) et delta sur le duel publicitaire.


## Exercice 3 : l'évanouissement mesuré — annonce révocable et caution minimale

Toujours sur le duel publicitaire : (a) codez l'arbre à trois étages « A annonce, B choisit, A choisit librement » et montrez par induction à rebours que l'issue est **indépendante de l'annonce** — le gain d'engagement s'évanouit ; (b) introduisez la caution `s` qui pénalise l'écart de promesse de A, balayez `s` et trouvez le seuil `s*` où l'engagement redevient crédible ; (c) vérifiez que `s*` égale l'écart de tentation de A sur la colonne critique. Sans (c), la félicité n'a pas été mesurée.

```python
# TODO etudiant
# Etape 1 : arbre a 3 etages + induction a rebours (modele : cellule 9).
# Etape 2 : caution s sur l ecart de promesse, balayage (modele : cellule 11).
# Etape 3 : s* contre l ecart de tentation -- l identite qui ferme l exercice.
result_exo3 = None
print('Exercice a completer : evanouissement + caution minimale sur le duel publicitaire.')
```

In [8]:
# EXERCICE 3 : annonce revocable (issue independante de l annonce) puis caution minimale s*.
# Indice : l arbre de la cellule 9 se transpose ; la caution modifie UN seul payoff (l ecart de promesse).
# Etape 1 : induction a rebours sur l arbre a 3 etages -> issue identique pour toute annonce.
# Etape 2 : balayage de s -> tableau issue/u_A (modele cellule 11).
# Etape 3 : s* = ecart de tentation sur la colonne critique -> l egaler par le calcul.
result_exo3 = None  # TODO etudiant
print('Exercice a completer : evanouissement + caution minimale sur le duel publicitaire.')

Exercice a completer : evanouissement + caution minimale sur le duel publicitaire.


## Résumé : quatre régimes, un seul nombre qui change, et ce qu'il prouve

| régime | mécanisme | issue | u_I |
|---|---|---|---|
| simultané, équilibre crédible | Nash `(In, Acc)` | entrée accueillie | 1 |
| simultané, équilibre de menace | Nash `(Out, Fight)` — incroyable | dissuasion *non tenable* | (3, fantôme) |
| séquentiel sans engagement | induction à rebours | `(In, Acc)` — la menace dissoute | 1 |
| **engagement contraignant** | `sigma_L -> BR_F(sigma_L)` | **`(Out, Fight)` — sélectionné** | **3** |
| annonce révocable | cheap talk — arbre à 3 étages | `(In, Acc)` — indépendant de l'annonce | 1 |
| engagement + caution s ≥ 3 | dévier ne rapporte plus | `(Out, Fight)` tenu sans jamais combattre | 3 |

Le noyau tient en trois phrases. **Un engagement n'invente pas de valeur : il sélectionne un équilibre existant** que la menace seule ne pouvait pas tenir — le delta +2 était déjà dans la matrice, endormi dans la colonne `Out`. **L'acte engagé vaut par la modification de l'espace de réponse d'autrui**, non par son résultat direct : `Fight` vaut −2 *exécutée* contre une entrée, et 3 *commise* contre l'entrée qui n'aura pas lieu. **La félicité se mesure** : une annonce sans lien ne fait rien (l'issue est indépendante de l'annonce), et restaurer le lien coûte exactement l'écart de tentation `s* = 3` — la caution minimale telle que dévier ne rapporte plus.

C'est une performativité sans mystère au sens strict : la chaîne `représentation publique -> anticipation collective -> changement des actions -> nouvelle réalité stratégique` est ici un calcul réversible, chaque flèche étant une cellule de ce notebook. Ce que le cadre ne couvre **pas** — et il faut l'écrire — : les contextes où l'engagement est multi-décisionnel et répété (réputation dynamique), où la caution est endogène (capacités détruites plutôt que caution versée), et les jeux d'inspection où c'est l'engagement *mixte* qui bat le sommet pur. Chacun de ces reliefs est un notebook à part entière — pas une extrapolation sociologique de celui-ci.

**Ouvertures de la série** : la sélection d'équilibre par contrainte rejoint les [jeux répétés et le folk theorem](GameTheory-06b-Lean-RepeatedGames.ipynb) (l'engagement comme cas limite de la punition crédible), et la modification de l'espace des actions plutôt que du choix en leur sein est exactement le geste de la strate 7 sur les vocabulaires — voir aussi GT-18, les open games (issue #12212, en cours de livraison), où la représentation locale modifie le global dont elle est issue.